In [ ]:
# ================================================
# Deping 프로젝트 - TMDB 포스터 로컬 다운로드
# 파일명: 01_tmdb_collect.ipynb (포스터 다운로드 파트)
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

In [1]:
# ================================================
# Deping 프로젝트 - TMDB 포스터 로컬 다운로드
# 파일명: 01_tmdb_collect.ipynb (포스터 다운로드 파트)
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

# 1. .env 파일 로드 (TMDB_API_KEY 사용)
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    raise ValueError("❌ .env 파일에 TMDB_API_KEY가 없습니다. .env 파일을 확인해주세요!")

print("✅ TMDB API Key 로드 완료")

# 2. 저장 폴더 생성
POSTER_DIR = Path("data/posters")
POSTER_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ 저장 폴더 생성: {POSTER_DIR.resolve()}")

# 3. 포스터 다운로드 함수
def download_poster(
    poster_path: str,
    movie_title: str,
    size: str = "w500",
    max_retries: int = 3
) -> str | None:
    """TMDB 포스터를 로컬에 다운로드하고 파일 경로 반환"""
    if not poster_path:
        return None
    
    url = f"https://image.tmdb.org/t/p/{size}{poster_path}"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, stream=True, timeout=15)
            
            if response.status_code == 200:
                # 안전한 파일명 만들기
                safe_title = "".join(c for c in movie_title if c.isalnum() or c in " _-").strip()
                if not safe_title:
                    safe_title = f"movie_{int(time.time())}"
                
                file_path = POSTER_DIR / f"{safe_title}_poster.jpg"
                
                with open(file_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024*1024):  # 1MB씩
                        f.write(chunk)
                
                return str(file_path)
                
            elif response.status_code == 429:  # Rate limit
                print(f"⚠️ Rate limit ({attempt+1}/{max_retries})")
                time.sleep(2)
                continue
                
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ 다운로드 실패: {movie_title} → {e}")
            time.sleep(1)
    
    return None

✅ TMDB API Key 로드 완료
✅ 저장 폴더 생성: D:\deping\notebooks\data\posters


In [2]:
# ================================================
# 4. 영화 데이터 불러오기 (두 가지 방식 선택)
# ================================================

# ===== 방식 A: 기존 JSON 파일이 있다면 (추천) =====
data_file = Path("data/raw/tmdb_movies.json")

if data_file.exists():
    with open(data_file, "r", encoding="utf-8") as f:
        movies = json.load(f)
    print(f"✅ 기존 JSON에서 {len(movies)}개 영화 로드 완료")
else:
    # ===== 방식 B: TMDB API에서 새로 수집 (처음 실행할 때) =====
    print("📥 TMDB API에서 인기 영화 수집 중...")
    movies = []
    pages = 5  # 5페이지 ≈ 100개 영화 (원하는 만큼 늘려도 OK)
    
    for page in tqdm(range(1, pages + 1), desc="TMDB API 호출"):
        url = f"https://api.themoviedb.org/3/movie/popular"
        params = {
            "api_key": TMDB_API_KEY,
            "language": "ko-KR",
            "page": page
        }
        resp = requests.get(url, params=params, timeout=10)
        
        if resp.status_code == 200:
            data = resp.json()
            for movie in data.get("results", []):
                if movie.get("poster_path"):  # 포스터 있는 영화만
                    movies.append({
                        "tmdb_id": movie["id"],
                        "title": movie["title"],
                        "title_ko": movie.get("title", movie["original_title"]),
                        "poster_path": movie["poster_path"],
                        "overview": movie.get("overview", ""),
                        "genres": [],  # 나중에 enrich 가능
                        "year": movie.get("release_date", "")[:4]
                    })
        time.sleep(0.3)  # Rate limit 방지
    
    print(f"✅ API에서 {len(movies)}개 영화 수집 완료")


📥 TMDB API에서 인기 영화 수집 중...


TMDB API 호출: 100%|██████████| 5/5 [00:05<00:00,  1.07s/it]

✅ API에서 100개 영화 수집 완료


In [3]:
# ================================================
# 5. 포스터 일괄 다운로드 (진행률 표시)
# ================================================

print(f"\n🚀 {len(movies)}개 영화 포스터 다운로드 시작...")

for movie in tqdm(movies, desc="포스터 다운로드"):
    local_path = download_poster(
        poster_path=movie.get("poster_path"),
        movie_title=movie.get("title_ko") or movie.get("title")
    )
    
    # JSON에 로컬 경로 추가 (나중에 Azure AI Search에 업로드할 때 사용)
    if local_path:
        movie["poster_local_path"] = local_path
    else:
        movie["poster_local_path"] = None


🚀 100개 영화 포스터 다운로드 시작...


포스터 다운로드: 100%|██████████| 100/100 [01:05<00:00,  1.52it/s]


In [ ]:





# ================================================
# 6. 결과 저장
# ================================================

# 업데이트된 JSON 저장
output_file = Path("data/processed/tmdb_movies_with_posters.json")
output_file.parent.mkdir(parents=True, exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print(f"\n🎉 완료!")
print(f"   • 다운로드된 포스터 개수: {sum(1 for m in movies if m.get('poster_local_path'))}개")
print(f"   • 저장 위치: {POSTER_DIR.resolve()}")
print(f"   • 업데이트된 JSON: {output_file.resolve()}")

In [4]:
# ================================================
# Deping 프로젝트 — 한국에서 볼 수 있는 영화 데이터 수집
# 파일명: 01_tmdb_collect.ipynb
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

# 1. .env 로드
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    raise ValueError("❌ .env에 TMDB_API_KEY가 없습니다!")

print("✅ TMDB API Key 로드 완료")

# 2. 저장 폴더 생성
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. Discover API로 한국 필터링 + 한국어 데이터 가져오기
def fetch_korean_movies(pages: int = 15):
    """한국에서 볼 수 있는 고품질 영화 수집"""
    movies = []
    base_url = "https://api.themoviedb.org/3/discover/movie"
    
    print(f"📥 TMDB Discover API 호출 중... (한국 region=KR, language=ko-KR, {pages}페이지)")
    
    for page in tqdm(range(1, pages + 1), desc="페이지 진행"):
        params = {
            "api_key": TMDB_API_KEY,
            "language": "ko-KR",           # ← 한국어 제목 + 줄거리
            "region": "KR",                # ← 한국 개봉/릴리즈 정보 필터
            "sort_by": "popularity.desc",  # 인기순
            "vote_average.gte": 6.5,       # 평점 6.5 이상
            "vote_count.gte": 1000,        # 투표수 1,000 이상 (데이터 신뢰도 ↑)
            "include_adult": False,
            "page": page,
            "with_original_language": "ko|en|ja|zh|fr|es"  # 한국 + 주요 언어
        }
        
        resp = requests.get(base_url, params=params, timeout=15)
        
        if resp.status_code == 200:
            data = resp.json()
            for movie in data.get("results", []):
                if movie.get("poster_path") and movie.get("overview"):  # 포스터 + 줄거리 있는 것만
                    movies.append({
                        "tmdb_id": movie["id"],
                        "title": movie["title"],                    # 원문 제목
                        "title_ko": movie.get("title", movie["original_title"]),  # 한국어 제목
                        "poster_path": movie["poster_path"],
                        "poster_url": f"https://image.tmdb.org/t/p/w500{movie['poster_path']}",  # ← 바로 사용 가능
                        "overview": movie.get("overview", ""),      # 한국어 줄거리
                        "release_date": movie.get("release_date"),
                        "vote_average": movie.get("vote_average"),
                        "vote_count": movie.get("vote_count"),
                        "genres": [],  # 나중에 enrich 가능
                        "year": movie.get("release_date", "")[:4] if movie.get("release_date") else None
                    })
        else:
            print(f"⚠️ 페이지 {page} 오류: {resp.status_code}")
        
        time.sleep(0.25)  # Rate limit 안전
    
    print(f"✅ 총 {len(movies)}개 한국 필터링 영화 수집 완료!")
    return movies

# 4. 데이터 수집 실행
movies = fetch_korean_movies(pages=15)   # 원하시면 20~30으로 늘려도 OK

# 5. 결과 저장
output_raw = RAW_DIR / "tmdb_movies_kr_filtered.json"
output_processed = PROCESSED_DIR / "tmdb_movies_kr_with_poster.json"

with open(output_raw, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

with open(output_processed, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print("\n🎉 완료!")
print(f"   • 수집된 영화 수: {len(movies)}개")
print(f"   • 한국어 줄거리 포함: Yes")
print(f"   • poster_url 자동 생성: Yes (다운로드 없이 바로 사용 가능)")
print(f"   • 저장 파일:")
print(f"     - {output_raw.resolve()}")
print(f"     - {output_processed.resolve()}")

✅ TMDB API Key 로드 완료
📥 TMDB Discover API 호출 중... (한국 region=KR, language=ko-KR, 15페이지)


페이지 진행: 100%|██████████| 15/15 [00:15<00:00,  1.05s/it]

✅ 총 300개 한국 필터링 영화 수집 완료!

🎉 완료!
   • 수집된 영화 수: 300개
   • 한국어 줄거리 포함: Yes
   • poster_url 자동 생성: Yes (다운로드 없이 바로 사용 가능)
   • 저장 파일:
     - D:\deping\notebooks\data\raw\tmdb_movies_kr_filtered.json
     - D:\deping\notebooks\data\processed\tmdb_movies_kr_with_poster.json


In [6]:
# ================================================
# Deping 프로젝트 — 한국 필터링 + 감독/배우/키워드 포함 데이터 수집
# 파일명: 01_tmdb_collect.ipynb
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

# 1. .env 로드
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    raise ValueError("❌ .env에 TMDB_API_KEY가 없습니다!")

print("✅ TMDB API Key 로드 완료")

# 2. 저장 폴더 생성
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. Credits API (감독 + 배우)
def fetch_credits(movie_id: int):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    params = {"api_key": TMDB_API_KEY, "language": "ko-KR"}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            director = next((crew["name"] for crew in data.get("crew", []) if crew.get("job") == "Director"), None)
            actors = [cast["name"] for cast in data.get("cast", [])[:8]]
            return director, actors
        return None, []
    except Exception:
        return None, []

# 4. Keywords API (새로 추가)
def fetch_keywords(movie_id: int):
    """영화 키워드 리스트 가져오기"""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    params = {"api_key": TMDB_API_KEY}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            keywords = [k["name"] for k in data.get("keywords", [])]
            return keywords
        return []
    except Exception:
        return []

# 5. Discover API로 한국 필터링 영화 수집
def fetch_korean_movies(pages: int = 25):
    movies = []
    base_url = "https://api.themoviedb.org/3/discover/movie"
    
    print(f"📥 TMDB Discover API 호출 중... ({pages}페이지, 한국 필터)")
    
    for page in tqdm(range(1, pages + 1), desc="기본 영화 목록 수집"):
        params = {
            "api_key": TMDB_API_KEY,
            "language": "ko-KR",
            "region": "KR",
            "sort_by": "popularity.desc",
            "vote_average.gte": 6.5,
            "vote_count.gte": 1000,
            "include_adult": False,
            "page": page,
        }
        
        resp = requests.get(base_url, params=params, timeout=15)
        
        if resp.status_code == 200:
            data = resp.json()
            for movie in data.get("results", []):
                if movie.get("poster_path") and movie.get("overview"):
                    movies.append({
                        "tmdb_id": movie["id"],
                        "title": movie["title"],
                        "title_ko": movie.get("title", movie["original_title"]),
                        "poster_path": movie["poster_path"],
                        "poster_url": f"https://image.tmdb.org/t/p/w500{movie['poster_path']}",
                        "overview": movie.get("overview", ""),
                        "release_date": movie.get("release_date"),
                        "vote_average": movie.get("vote_average"),
                        "vote_count": movie.get("vote_count"),
                        "year": movie.get("release_date", "")[:4] if movie.get("release_date") else None,
                        # 아래 필드들은 enrichment에서 채움
                        "director": None,
                        "actors": [],
                        "keywords": []          # ← 새로 추가
                    })
        time.sleep(0.25)
    
    return movies

# 6. 데이터 수집 + Credits + Keywords 보강
print("🚀 1단계: 한국 필터링 영화 목록 수집")
movies = fetch_korean_movies(pages=25)   # 필요하면 15~20으로 늘려도 OK

print(f"\n🚀 2단계: {len(movies)}개 영화에 감독/배우/키워드 보강 중...")

for movie in tqdm(movies, desc="Credits & Keywords 보강"):
    # 감독 + 배우
    director, actors = fetch_credits(movie["tmdb_id"])
    movie["director"] = director
    movie["actors"] = actors
    
    # 키워드
    keywords = fetch_keywords(movie["tmdb_id"])
    movie["keywords"] = keywords
    
    time.sleep(0.35)   # Rate limit 안전 (3개 API 호출)

# 7. 결과 저장
output_raw = RAW_DIR / "tmdb_movies_kr_with_credits_keywords2.json"
output_processed = PROCESSED_DIR / "tmdb_movies_kr_with_credits_keywords2.json"

with open(output_raw, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

with open(output_processed, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

# 8. 완료 메시지 + 샘플 출력
print("\n🎉 모든 작업 완료!")
print(f"   • 총 영화 수: {len(movies)}개")
print(f"   • 포함된 정보: 한국어 줄거리 + poster_url + 감독 + 배우(상위 8명) + 키워드")
print(f"   • 저장 파일:")
print(f"     - {output_raw.resolve()}")
print(f"     - {output_processed.resolve()}")

# 첫 번째 영화 샘플 확인
if movies:
    sample = movies[0]
    print("\n📋 첫 번째 영화 샘플:")
    print(f"   제목: {sample['title_ko']}")
    print(f"   감독: {sample['director']}")
    print(f"   배우: {sample['actors'][:5]} ...")
    print(f"   키워드: {sample['keywords'][:8]} ...")   # 키워드도 출력

✅ TMDB API Key 로드 완료
🚀 1단계: 한국 필터링 영화 목록 수집
📥 TMDB Discover API 호출 중... (25페이지, 한국 필터)


기본 영화 목록 수집: 100%|██████████| 25/25 [00:25<00:00,  1.01s/it]



🚀 2단계: 500개 영화에 감독/배우/키워드 보강 중...


Credits & Keywords 보강: 100%|██████████| 500/500 [15:08<00:00,  1.82s/it]


🎉 모든 작업 완료!
   • 총 영화 수: 500개
   • 포함된 정보: 한국어 줄거리 + poster_url + 감독 + 배우(상위 8명) + 키워드
   • 저장 파일:
     - D:\deping\notebooks\data\raw\tmdb_movies_kr_with_credits_keywords2.json
     - D:\deping\notebooks\data\processed\tmdb_movies_kr_with_credits_keywords2.json

📋 첫 번째 영화 샘플:
   제목: 아바타: 불과 재
   감독: 제임스 카메론
   배우: ['샘 워싱턴', '조 샐다나', '시고니 위버', '스티븐 랭', '우나 채플린'] ...
   키워드: ['witch', 'clone', 'space war', 'tribe', 'sequel', 'alien', 'transhumanism', 'family'] ...
